# Per-Day Decimation Encoding Analysis Orchestrator

Runs the same Poisson GLM decimation analysis as `decimation_orchestrator.ipynb` but restricted
to words spoken within a single calendar day (active hours 09:00–23:00 local time).

One SLURM job is submitted per **(patient, date, portion, sample)** combination.
Outputs land in `{vad_root}/{patient}/encoding/decimation_cv_per_day/{date}/`.

**Two-phase submission:**
1. **Phase 1** — one full-data job (portion=1.0, sample=0) per (patient, day). Runs inner-CV alpha
   tuning over that day's filtered words and saves `{patient}_best_alphas.npy`.
2. **Phase 2** — all other (portion, sample) jobs per (patient, day) with `--dependency=afterok` on
   the Phase 1 job and `--fixed-alpha-path` pointing to the saved alphas. No inner CV.

**Workflow:**
1. Run config + imports
2. Run **Compute per-day word indices** cell
3. Run **Phase 1** cell
4. Run **Phase 2** cell immediately — SLURM holds jobs until Phase 1 completes
5. Re-run status cell periodically
6. Run assembly cells once all jobs complete

In [3]:
from pathlib import Path
import subprocess
import json

import dill as pickle
import numpy as np
import pandas as pd
import torch

In [9]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PROJ_DIR    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT    = PROJ_DIR / 'vad_new'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                   '/decimation_analysis/decimation_glm_worker.py')
PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python'

PATIENTS       = None   # None => scan all Y* patient folders under vad_new
FORCE_RESUBMIT = False

# ── Day-partition settings ─────────────────────────────────────────────────────
LOCAL_TZ           = 'America/Chicago'
ACTIVE_HOURS_START = 9    # inclusive, local time
ACTIVE_HOURS_END   = 23   # exclusive, local time
MIN_WORDS_PER_DAY  = 200  # skip days with fewer quality-filtered words

# ── Decimation grid — must match original analysis ────────────────────────────
# PORTIONS[0] must be 1.0 (full data); N_SAMPLES[0] must be 1
PORTIONS  = [1.0,  0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_SAMPLES = [1,   100,  20,   10,  8,   7,   6,   5,   4,   3,   2,   2  ]

# ── Model settings ─────────────────────────────────────────────────────────────
SPIKE_OFFSET_IDX = 0
GPT2_LAYER       = -1
N_PCA            = 100
OUTER_SPLITS     = 5
INNER_SPLITS     = 5
N_ALPHAS         = 30
ALPHA_LOW        = -3.0
ALPHA_HIGH       = 3.0
N_SHUFFLES       = 50


# ── Compute mode ──────────────────────────────────────────────────────────────
USE_GPU = False

# ── SLURM settings ─────────────────────────────────────────────────────────────
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'
if USE_GPU:
    PARTITION = 'Universal'
    GRES      = 'gpu:1'
    CPUS      = 8
    N_PERM_JOBS      = CPUS * 2
else:
    PARTITION = 'Universal'
    GRES      = ''
    CPUS      = 16
    N_PERM_JOBS      = CPUS * 2

RUN_NAME            = 'decimation_cv_per_day'
GLOBAL_LOGS_DIR     = VAD_ROOT / f'{RUN_NAME}_slurm_logs'
GLOBAL_SCRIPTS_DIR  = VAD_ROOT / f'{RUN_NAME}_slurm_scripts'
GLOBAL_LOGS_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('vad root:    ', VAD_ROOT)
print('worker:      ', WORKER_PATH)
print('logs dir:    ', GLOBAL_LOGS_DIR)
print('scripts dir: ', GLOBAL_SCRIPTS_DIR)
print('portions:    ', PORTIONS)
print('n_samples:   ', N_SAMPLES)
print(f'compute:      {"GPU" if USE_GPU else "CPU"} | partition={PARTITION}')
print(f'active hours: {ACTIVE_HOURS_START}:00 \u2013 {ACTIVE_HOURS_END}:00 {LOCAL_TZ}')

vad root:     /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new
worker:       /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/decimation_analysis/decimation_glm_worker.py
logs dir:     /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/decimation_cv_per_day_slurm_logs
scripts dir:  /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/decimation_cv_per_day_slurm_scripts
portions:     [1.0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
n_samples:    [1, 100, 20, 10, 8, 7, 6, 5, 4, 3, 2, 2]
compute:      CPU | partition=Universal
active hours: 9:00 – 23:00 America/Chicago


## Compute per-day filtered word indices

Reads `{patient}_word_df_filtered.csv`, converts UTC timestamps to local time, and saves one
`.npy` index file per qualifying day. Only words within active hours and days with at least
`MIN_WORDS_PER_DAY` quality-filtered words are kept.

In [5]:
def compute_day_indices(patient: str) -> 'dict[str, tuple[Path, int]]':
    """
    Load quality-filtered word df, group by local calendar date within active hours,
    and save one .npy index file per qualifying day.
    Returns {date_str: (idx_path, n_words)}.
    """
    patient_root = VAD_ROOT / patient
    filtered_csv = patient_root / f'{patient}_word_df_filtered.csv'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return {}

    df = pd.read_csv(filtered_csv, usecols=['original_word_idx', 'utc_word_start'])
    df['utc_word_start'] = pd.to_datetime(df['utc_word_start'], utc=True)
    df['local_dt'] = df['utc_word_start'].dt.tz_convert(LOCAL_TZ)
    df['hour']     = df['local_dt'].dt.hour
    df['date_str'] = df['local_dt'].dt.strftime('%Y-%m-%d')

    active = df[(df['hour'] >= ACTIVE_HOURS_START) & (df['hour'] < ACTIVE_HOURS_END)]

    day_paths = {}
    for date_str, grp in active.groupby('date_str'):
        if len(grp) < MIN_WORDS_PER_DAY:
            print(f'  [{patient}] {date_str}: only {len(grp)} words \u2014 skipping')
            continue
        idx_dir  = patient_root / 'encoding' / RUN_NAME / date_str
        idx_dir.mkdir(parents=True, exist_ok=True)
        idx_path = idx_dir / f'{patient}_{date_str}_word_idx.npy'
        word_idx = grp['original_word_idx'].values.astype(np.int64)
        np.save(idx_path, word_idx)
        day_paths[date_str] = (idx_path, len(word_idx))
        print(f'  [{patient}] {date_str}: {len(word_idx):,} words \u2192 {idx_path.name}')

    return day_paths


if PATIENTS is None:
    all_patients = sorted(p.name for p in VAD_ROOT.iterdir() if p.is_dir() and p.name.startswith('Y'))
else:
    all_patients = list(PATIENTS)

print(f'Computing per-day word indices for {len(all_patients)} patients...\n')
patient_day_idx = {}   # {patient: {date_str: (idx_path, n_words)}}
for patient in all_patients:
    day_data = compute_day_indices(patient)
    if day_data:
        patient_day_idx[patient] = day_data

total_pairs = sum(len(v) for v in patient_day_idx.values())
print(f'\nTotal (patient, day) pairs ready: {total_pairs}')

Computing per-day word indices for 21 patients...

  [YEY] 2024-04-01: 16,510 words → YEY_2024-04-01_word_idx.npy
  [YEY] 2024-04-02: 23,841 words → YEY_2024-04-02_word_idx.npy
  [YEZ] 2024-04-09: 6,698 words → YEZ_2024-04-09_word_idx.npy
  [YEZ] 2024-04-10: 16,464 words → YEZ_2024-04-10_word_idx.npy
  [YEZ] 2024-04-11: 1,584 words → YEZ_2024-04-11_word_idx.npy
  [YEZ] 2024-04-12: 7,996 words → YEZ_2024-04-12_word_idx.npy
  [YEZ] 2024-04-14: 18,298 words → YEZ_2024-04-14_word_idx.npy
  [YEZ] 2024-04-15: 14,595 words → YEZ_2024-04-15_word_idx.npy
  [YFA] 2024-04-23: 18,729 words → YFA_2024-04-23_word_idx.npy
  [YFA] 2024-04-24: 58,808 words → YFA_2024-04-24_word_idx.npy
  [YFA] 2024-04-25: 48,473 words → YFA_2024-04-25_word_idx.npy
  [YFA] 2024-04-26: 36,215 words → YFA_2024-04-26_word_idx.npy
  [YFA] 2024-04-27: 66,950 words → YFA_2024-04-27_word_idx.npy
  [YFA] 2024-04-28: 43,939 words → YFA_2024-04-28_word_idx.npy
  [YFA] 2024-04-29: 54,695 words → YFA_2024-04-29_word_idx.npy
  [YFA]

## Helper functions

In [6]:
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def compute_resources(n_words: int, full_cv: bool = False) -> dict:
    """
    Return adaptive SLURM mem/time based on estimated word count.
    full_cv=True applies a 3x time multiplier to account for inner-CV alpha tuning.
    """
    if USE_GPU:
        if n_words >= 20_000:
            base = dict(gres='mps:l40:100', mem='32G', time='24:00:00')
        elif n_words >= 10_000:
            base = dict(gres='mps:l40:50',  mem='16G', time='12:00:00')
        elif n_words >= 3_000:
            base = dict(gres='mps:l40:33',  mem='12G', time='8:00:00')
        else:
            base = dict(gres='mps:l40:10',  mem='8G',  time='4:00:00')
    else:
        if n_words >= 20_000:
            base = dict(gres='', mem='16G', time='48:00:00')
        elif n_words >= 10_000:
            base = dict(gres='', mem='12G', time='24:00:00')
        elif n_words >= 3_000:
            base = dict(gres='', mem='8G',  time='16:00:00')
        else:
            base = dict(gres='', mem='4G',  time='8:00:00')

    if full_cv:
        h        = int(base['time'].split(':')[0])
        tail     = ':'.join(base['time'].split(':')[1:])
        new_h    = min(h * 3, 72)
        base     = dict(base, time=f'{new_h:02d}:{tail}')
    return base


def resolve_patient_base_paths(patient: str) -> dict:
    patient_root = VAD_ROOT / patient
    embeddings_path = first_existing([
        patient_root / 'embeddings' / f'{patient}_gpt2_embeddings.npy',
        patient_root / 'all_convo_recording' / 'all_words_filtered_all_layers_gpt2.npy',
    ])
    counts_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_spike_counts_offsets_all.npy',
        patient_root / 'all_convo_recording' / 'word_spike_counts_offsets_all.npy',
    ])
    durations_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_durs.npy',
        patient_root / 'all_convo_recording' / 'word_durs.npy',
    ])
    return dict(
        patient_root=patient_root,
        embeddings_path=embeddings_path,
        counts_path=counts_path,
        durations_path=durations_path,
    )


def job_tag(portion, sample):
    return f'portion{portion}_sample{sample}'


def job_paths(out_dir: Path, patient: str, portion: float, sample: int) -> dict:
    tag = job_tag(portion, sample)
    return dict(
        success = out_dir / f'{patient}_{tag}_SUCCESS',
        error   = out_dir / f'{patient}_{tag}_error.txt',
        pkl     = out_dir / f'{patient}_{tag}_encoding_results_cv.pkl',
        tar     = out_dir / f'{patient}_{tag}_encoding_models_cv.tar',
        meta    = out_dir / f'{patient}_{tag}_meta.json',
    )


def build_combo_list() -> list:
    """One dict per (patient, day) pair with all resolved paths."""
    combos = []
    for patient in all_patients:
        day_data = patient_day_idx.get(patient, {})
        if not day_data:
            continue
        base = resolve_patient_base_paths(patient)
        ready = (base['embeddings_path'] is not None and
                 base['counts_path'] is not None and
                 base['durations_path'] is not None)
        for date_str, (idx_path, n_words) in sorted(day_data.items()):
            out_dir    = base['patient_root'] / 'encoding' / RUN_NAME / date_str
            alpha_path = out_dir / f'{patient}_best_alphas.npy'
            combos.append(dict(
                patient=patient,
                date_str=date_str,
                out_dir=out_dir,
                word_idx_path=idx_path,
                embeddings_path=base['embeddings_path'],
                counts_path=base['counts_path'],
                durations_path=base['durations_path'],
                alpha_path=alpha_path,
                n_day_words=n_words,
                ready_inputs=ready,
            ))
    return combos


def build_pair_status(combos: list) -> pd.DataFrame:
    """Summary table: one row per (patient, day)."""
    rows = []
    for c in combos:
        full_p    = job_paths(c['out_dir'], c['patient'], 1.0, 0)
        done_jobs = sum(
            1 for p_idx, p in enumerate(PORTIONS)
            for s in range(N_SAMPLES[p_idx])
            if job_paths(c['out_dir'], c['patient'], p, s)['success'].exists()
        )
        rows.append(dict(
            patient=c['patient'],
            date=c['date_str'],
            n_day_words=c['n_day_words'],
            ready_inputs=c['ready_inputs'],
            full_done=full_p['success'].exists(),
            full_error=full_p['error'].exists(),
            has_best_alphas=c['alpha_path'].exists(),
            jobs_done=done_jobs,
            jobs_total=sum(N_SAMPLES),
        ))
    return pd.DataFrame(rows).sort_values(['patient', 'date']).reset_index(drop=True)


def build_job_status(combos: list) -> pd.DataFrame:
    """Flat table: one row per (patient, day, portion, sample)."""
    rows = []
    for c in combos:
        if not c['ready_inputs']:
            continue
        for p_idx, portion in enumerate(PORTIONS):
            for sample in range(N_SAMPLES[p_idx]):
                paths = job_paths(c['out_dir'], c['patient'], portion, sample)
                rows.append(dict(
                    patient=c['patient'],
                    date=c['date_str'],
                    portion=portion,
                    sample=sample,
                    is_full=(portion == 1.0 and sample == 0),
                    done=(
                        paths['success'].exists() and
                        paths['meta'].exists() and
                        json.loads(paths['meta'].read_text()).get('n_shuffles', 0) > 0
                    ),
                    error=paths['error'].exists(),
                    has_pkl=paths['pkl'].exists(),
                ))
    return pd.DataFrame(rows)


print('helpers defined')

helpers defined


In [7]:
# ── (patient, day) pair overview ───────────────────────────────────────────────
combos      = build_combo_list()
pair_status = build_pair_status(combos)
print(f'(patient, day) pairs total: {len(pair_status)}')
print(f'ready inputs:               {int(pair_status["ready_inputs"].sum())}')
print(f'full jobs done:             {int(pair_status["full_done"].sum())}')
print(f'have best_alphas:           {int(pair_status["has_best_alphas"].sum())}')
display(pair_status[['patient', 'date', 'n_day_words', 'ready_inputs',
                      'full_done', 'full_error', 'has_best_alphas', 'jobs_done', 'jobs_total']])

(patient, day) pairs total: 156
ready inputs:               156
full jobs done:             0
have best_alphas:           0


,patient,date,n_day_words,ready_inputs,full_done,full_error,has_best_alphas,jobs_done,jobs_total
0,YEY,2024-04-01,16510,True,False,True,False,0,168
1,YEY,2024-04-02,23841,True,False,True,False,0,168
2,YEZ,2024-04-09,6698,True,False,True,False,0,168
3,YEZ,2024-04-10,16464,True,False,True,False,0,168
4,YEZ,2024-04-11,1584,True,False,True,False,0,168
...,...,...,...,...,...,...,...,...,...
151,YFU,2025-12-18,16053,True,False,True,False,0,168
152,YFV,2026-02-17,17708,True,False,True,False,0,168
153,YFV,2026-02-18,13530,True,False,True,False,0,168
154,YFV,2026-02-19,32670,True,False,True,False,0,168


## Phase 1: Submit full-data (portion=1.0) jobs

One job per (patient, day). Runs inner-CV alpha tuning over that day's filtered words
and saves `{patient}_best_alphas.npy` for use by all portion jobs.

In [ ]:
full_job_ids   = {}   # {(patient, date_str): slurm_job_id}  — used by Phase 2
submitted_full = []
skipped_full   = []

for c in combos:
    patient  = c['patient']
    date_str = c['date_str']
    if not c['ready_inputs']:
        continue

    full_p = job_paths(c['out_dir'], patient, 1.0, 0)

    if full_p['success'].exists() and not FORCE_RESUBMIT:
        skipped_full.append((patient, date_str))
        full_job_ids[(patient, date_str)] = 'already_done'
        continue

    c['out_dir'].mkdir(parents=True, exist_ok=True)
    reset_line = f"rm -f {full_p['success']} {full_p['error']}" if FORCE_RESUBMIT else 'true'

    res = compute_resources(c['n_day_words'], full_cv=True)

    cmd = (
        f'{PYTHON_BIN} {WORKER_PATH} {patient} {VAD_ROOT} {c["out_dir"]} '
        f'--portion 1.0 --sample 0 '
        f'--word-idx-path {c["word_idx_path"]} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER} '
        f'--n-pca {N_PCA} '
        f'--outer-splits {OUTER_SPLITS} '
        f'--inner-splits {INNER_SPLITS} '
        f'--n-alphas {N_ALPHAS} '
        f'--alpha-low {ALPHA_LOW} '
        f'--alpha-high {ALPHA_HIGH} '
        f'--n-shuffles {N_SHUFFLES} '
        f'--n-perm-jobs {N_PERM_JOBS} '
        f'--embeddings-path {c["embeddings_path"]} '
        f'--counts-path {c["counts_path"]} '
        f'--durations-path {c["durations_path"]}'
    )

    safe_date  = date_str.replace('-', '')
    gres_lines = [f'#SBATCH --gres={res["gres"]}'] if res['gres'] else []
    lines = [
        '#!/bin/bash',
        f'#SBATCH --job-name=dcp_full_{patient}_{safe_date}',
        f'#SBATCH --partition={PARTITION}',
        *gres_lines,
        f'#SBATCH --cpus-per-task={CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={res["mem"]}',
        f'#SBATCH --time={res["time"]}',
        f'# n_day_words={c["n_day_words"]} (full CV)',
        f'#SBATCH --output={GLOBAL_LOGS_DIR}/{patient}_{date_str}_full_%j.out',
        f'#SBATCH --error={GLOBAL_LOGS_DIR}/{patient}_{date_str}_full_%j.err',
        '',
        'set -eo pipefail',
        CONDA_ACTIVATE,
        'echo "HOSTNAME: $(hostname)"',
        'echo "START: $(date)"',
        f'echo "PATIENT: {patient} | DATE: {date_str} | FULL DATA ({c["n_day_words"]:,} words)"',
        reset_line,
        cmd,
        'echo "END: $(date)"',
    ]

    sbatch_path = GLOBAL_SCRIPTS_DIR / f'{patient}_{date_str}_full.sbatch'
    sbatch_path.write_text('\n'.join(lines))

    try:
        sub_res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
        job_id  = sub_res.stdout.strip().split()[-1]
        full_job_ids[(patient, date_str)] = job_id
        submitted_full.append((patient, date_str, job_id))
        print(f'submitted full: {patient} {date_str} -> job {job_id}')
    except subprocess.CalledProcessError as exc:
        print(f'FAILED SUBMISSION: {patient} {date_str}\n{exc.stderr}')

print(f'\nsubmitted={len(submitted_full)}, skipped={len(skipped_full)}')
print('full_job_ids:', full_job_ids)

submitted full: YEY 2024-04-01 -> job 399889
submitted full: YEY 2024-04-02 -> job 399890
submitted full: YEZ 2024-04-09 -> job 399891
submitted full: YEZ 2024-04-10 -> job 399892
submitted full: YEZ 2024-04-11 -> job 399893
submitted full: YEZ 2024-04-12 -> job 399894
submitted full: YEZ 2024-04-14 -> job 399895
submitted full: YEZ 2024-04-15 -> job 399896
submitted full: YFA 2024-04-23 -> job 399897
submitted full: YFA 2024-04-24 -> job 399898
submitted full: YFA 2024-04-25 -> job 399899
submitted full: YFA 2024-04-26 -> job 399900
submitted full: YFA 2024-04-27 -> job 399901
submitted full: YFA 2024-04-28 -> job 399902
submitted full: YFA 2024-04-29 -> job 399903
submitted full: YFA 2024-04-30 -> job 399904
submitted full: YFA 2024-05-01 -> job 399905
submitted full: YFB 2024-05-02 -> job 399906
submitted full: YFB 2024-05-03 -> job 399907
submitted full: YFB 2024-05-04 -> job 399908
submitted full: YFB 2024-05-05 -> job 399909
submitted full: YFB 2024-05-06 -> job 399910
submitted 

## Phase 2: Submit portion jobs

One job per (patient, day, portion, sample) for all portions != 1.0.
Each job uses `--fixed-alpha-path` (skips inner CV) and `--dependency=afterok` if the
corresponding Phase 1 job hasn't finished yet.

Run this cell immediately after Phase 1 — SLURM holds the jobs until the dependency completes.

In [ ]:
submitted_portions = []

for c in combos:
    patient  = c['patient']
    date_str = c['date_str']
    if not c['ready_inputs']:
        continue

    c['out_dir'].mkdir(parents=True, exist_ok=True)

    full_done = job_paths(c['out_dir'], patient, 1.0, 0)['success'].exists()
    full_jid  = full_job_ids.get((patient, date_str), 'already_done')
    dep_str   = ''
    if not full_done and full_jid not in ('already_done', ''):
        dep_str = f'#SBATCH --dependency=afterok:{full_jid}'

    for p_idx, portion in enumerate(PORTIONS):
        if portion == 1.0:
            continue   # full run handled in Phase 1
        for sample in range(N_SAMPLES[p_idx]):
            paths = job_paths(c['out_dir'], patient, portion, sample)

            if paths['success'].exists() and not FORCE_RESUBMIT:
                continue

            n_words_est = max(1, int(round(portion * c['n_day_words'])))
            res         = compute_resources(n_words_est)

            reset_line = f"rm -f {paths['success']} {paths['error']}" if FORCE_RESUBMIT else 'true'

            cmd = (
                f'{PYTHON_BIN} {WORKER_PATH} {patient} {VAD_ROOT} {c["out_dir"]} '
                f'--portion {portion} --sample {sample} '
                f'--word-idx-path {c["word_idx_path"]} '
                f'--fixed-alpha-path {c["alpha_path"]} '
                f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
                f'--gpt2-layer {GPT2_LAYER} '
                f'--n-pca {N_PCA} '
                f'--outer-splits {OUTER_SPLITS} '
                f'--inner-splits {INNER_SPLITS} '
                f'--n-alphas {N_ALPHAS} '
                f'--alpha-low {ALPHA_LOW} '
                f'--alpha-high {ALPHA_HIGH} '
                f'--n-shuffles {N_SHUFFLES} '
                f'--n-perm-jobs {N_PERM_JOBS} '
                f'--embeddings-path {c["embeddings_path"]} '
                f'--counts-path {c["counts_path"]} '
                f'--durations-path {c["durations_path"]}'
            )

            tag       = job_tag(portion, sample)
            safe_date = date_str.replace('-', '')
            gres_lines = [f'#SBATCH --gres={res["gres"]}'] if res['gres'] else []
            lines = [
                '#!/bin/bash',
                f'#SBATCH --job-name=dcp_{patient}_{safe_date}_p{portion}',
                f'#SBATCH --partition={PARTITION}',
                *gres_lines,
                f'#SBATCH --cpus-per-task={CPUS}',
                f'#SBATCH --qos=big_batch_tier',
                f'#SBATCH --mem={res["mem"]}',
                f'#SBATCH --time={res["time"]}',
                f'# estimated n_words={n_words_est} (portion={portion} x n_day_words={c["n_day_words"]})',
                f'#SBATCH --output={GLOBAL_LOGS_DIR}/{patient}_{date_str}_{tag}_%j.out',
                f'#SBATCH --error={GLOBAL_LOGS_DIR}/{patient}_{date_str}_{tag}_%j.err',
            ]
            if dep_str:
                lines.append(dep_str)
            lines += [
                '',
                'set -eo pipefail',
                CONDA_ACTIVATE,
                'echo "HOSTNAME: $(hostname)"',
                'echo "START: $(date)"',
                f'echo "PATIENT: {patient} | DATE: {date_str} | portion={portion} sample={sample} (~{n_words_est} words)"',
                reset_line,
                cmd,
                'echo "END: $(date)"',
            ]

            sbatch_path = GLOBAL_SCRIPTS_DIR / f'{patient}_{date_str}_{tag}.sbatch'
            sbatch_path.write_text('\n'.join(lines))

            try:
                sub_res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
                job_id  = sub_res.stdout.strip().split()[-1]
                submitted_portions.append((patient, date_str, portion, sample, job_id))
            except subprocess.CalledProcessError as exc:
                print(f'FAILED: {patient} {date_str} {tag}\n{exc.stderr}')

print(f'submitted {len(submitted_portions)} portion jobs')

submitted 26052 portion jobs


## Check job status

In [ ]:
# ── (patient, day) summary ─────────────────────────────────────────────────────
combos      = build_combo_list()
pair_status = build_pair_status(combos)
print(f'(patient, day) pairs: {len(pair_status)}')
print(f'full jobs done:       {int(pair_status["full_done"].sum())} / {len(pair_status)}')
display(pair_status[['patient', 'date', 'n_day_words', 'full_done',
                      'full_error', 'has_best_alphas', 'jobs_done', 'jobs_total']])

In [ ]:
# ── Flat job status ─────────────────────────────────────────────────────────────
job_df = build_job_status(combos)
total  = len(job_df)
done   = int(job_df['done'].sum())
erred  = int(job_df['error'].sum())
print(f'total jobs: {total}  |  done: {done}  |  errors: {erred}  |  pending: {total - done - erred}')

by_pair = job_df.groupby(['patient', 'date']).agg(
    total=('done', 'count'),
    done=('done', 'sum'),
    errors=('error', 'sum'),
).reset_index()
display(by_pair)

## Inspect errors

In [ ]:
error_rows = job_df[job_df['error']].head(10)
print(f'jobs with error files: {len(job_df[job_df["error"]])}')

combo_lookup = {(c['patient'], c['date_str']): c for c in combos}
for _, row in error_rows.iterrows():
    c = combo_lookup.get((row['patient'], row['date']))
    if c is None:
        continue
    paths = job_paths(c['out_dir'], row['patient'], row['portion'], row['sample'])
    print('=' * 80)
    print(f"{row['patient']}  {row['date']}  portion={row['portion']}  sample={row['sample']}")
    if paths['error'].exists():
        print(paths['error'].read_text()[:2000])

## Assemble results

Load all completed PKL files, add `patient` and `date` columns, and concatenate into one DataFrame.

In [ ]:
# ── Load stats DataFrames ──────────────────────────────────────────────────────
records = []
loaded  = []

combo_lookup = {(c['patient'], c['date_str']): c for c in combos}

for _, row in job_df[job_df['has_pkl']].iterrows():
    c = combo_lookup.get((row['patient'], row['date']))
    if c is None:
        continue
    paths = job_paths(c['out_dir'], row['patient'], row['portion'], row['sample'])
    with open(paths['pkl'], 'rb') as f:
        df = pickle.load(f)
    df['patient'] = row['patient']
    df['date']    = row['date']
    records.append(df)
    loaded.append((row['patient'], row['date'], row['portion'], row['sample']))

if records:
    all_results = pd.concat(records, ignore_index=True)
    n_pairs = len(set((p, d) for p, d, _, _ in loaded))
    print(f'loaded {len(records)} result files across {n_pairs} (patient, day) pairs')
    print(f'total rows: {len(all_results)}')
    display(all_results.head())

    summary_rows = all_results[all_results['is_summary'] == True].copy()
    if 'pseudo_r2_mean' in summary_rows.columns:
        overview = summary_rows.groupby(['patient', 'date', 'portion']).agg(
            n_neurons=('neuron_idx', 'nunique'),
            n_samples=('sample', 'nunique'),
            r2_mean=('pseudo_r2_mean', 'mean'),
            pearson_mean=('pearson_corr_mean', 'mean'),
        ).reset_index()
        display(overview.sort_values(['patient', 'date', 'portion']))
else:
    print('No completed result PKLs found yet.')

In [ ]:
# ── Save assembled stats DataFrame ─────────────────────────────────────────────
if records:
    out_path = VAD_ROOT / f'{RUN_NAME}_results_all.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump(all_results, f)
    print(f'saved: {out_path}')

In [ ]:
# ── Assemble per-(patient, day) model tars ─────────────────────────────────────
# Merges all (portion, sample, fold) state_dicts within each day into one tar

done_pairs = sorted(set((p, d) for p, d, _, _ in loaded))

for patient, date_str in done_pairs:
    c = combo_lookup.get((patient, date_str))
    if c is None:
        continue

    pt_models  = {}
    pair_rows  = job_df[(job_df['patient'] == patient) & (job_df['date'] == date_str) & job_df['has_pkl']]

    for _, row in pair_rows.iterrows():
        paths    = job_paths(c['out_dir'], patient, row['portion'], row['sample'])
        tar_path = paths['tar']
        if not tar_path.exists():
            continue
        models = torch.load(tar_path, map_location='cpu')
        pt_models.update(models)

    if pt_models:
        out_tar = c['out_dir'] / f'{patient}_{date_str}_decimation_models_all.tar'
        torch.save(pt_models, out_tar)
        print(f'assembled {len(pt_models)} models -> {out_tar.name}')